# Week 2 Day 4: scoring after the swarm
First challenge from `labs/week2day4.ipynb`.

Add a `score_run` node after the parallel `work_claim` nodes and store its summary in the graph state. The flow is `list_claims -> work_claim (one per claim) -> score_run -> END`.

In [ ]:
import operator
import time
from collections.abc import Callable
from datetime import datetime, timedelta, timezone
from math import isclose
from typing import Annotated, Literal, TypedDict

import pandas as pd
from langgraph.graph import END, START, StateGraph
from langgraph.graph.state import CompiledStateGraph
from langgraph.types import RetryPolicy, Send

Decision = Literal["approve", "decline", "refer"]


class Answer(TypedDict):
    decision: Decision
    payout: float


class ClaimResult(TypedDict, total=False):
    claim_id: str
    started: datetime
    finished: datetime
    decision: Decision
    payout: float
    rationale: str
    cost: float


class RunScores(TypedDict, total=False):
    completed: int
    decision_accuracy: float
    accuracy: float
    cost_of_mistakes: float
    straight_through: float
    llm_cost: float
    seconds: float
    slowest_claim: float
    seconds_per_claim: float


class Dispatch(TypedDict, total=False):
    claims: list[str]
    results: Annotated[list[ClaimResult], operator.add]
    scores: RunScores

## Join the workers before scoring
The reducer combines each worker's result list. All `Send` tasks belong to the same worker step, even when `max_concurrency` limits how many execute at once. The shared `score_run` node runs in the next step with the complete list.

The builder accepts a worker and a scoring function, so the same graph supports the timing stand-in and the lab's full claim results. The `scores` field needs no reducer because only one node writes it.

In [ ]:
def build_dispatcher(
    claim_ids: list[str],
    worker: Callable[[str], Dispatch],
    scorer: Callable[[list[ClaimResult]], RunScores],
) -> CompiledStateGraph:
    """Collect every worker result, then score the run once."""
    def list_claims(state: Dispatch) -> Dispatch:
        return {"claims": claim_ids}

    def fan_out(state: Dispatch) -> list[Send]:
        return [Send("work_claim", claim_id) for claim_id in state["claims"]]

    def score_run(state: Dispatch) -> Dispatch:
        return {"scores": scorer(state["results"])}

    builder = StateGraph(Dispatch)
    builder.add_node("list_claims", list_claims)
    builder.add_node("work_claim", worker, retry_policy=RetryPolicy(max_attempts=3, retry_on=Exception))
    builder.add_node("score_run", score_run)
    builder.add_edge(START, "list_claims")
    builder.add_conditional_edges("list_claims", fan_out, ["work_claim"])
    builder.add_edge("work_claim", "score_run")
    builder.add_edge("score_run", END)
    return builder.compile()


def timing_scores(results: list[ClaimResult]) -> RunScores:
    """Measure completion and worker time for a nonempty run."""
    durations = [(row["finished"] - row["started"]).total_seconds() for row in results]
    return {
        "completed": len(results),
        "seconds": (max(row["finished"] for row in results) - min(row["started"] for row in results)).total_seconds(),
        "slowest_claim": max(durations),
        "seconds_per_claim": sum(durations) / len(durations),
    }


def stand_in(claim_id: str) -> Dispatch:
    """Simulate work with unequal durations, including a slower final claim."""
    started = datetime.now(timezone.utc)
    time.sleep(0.01 * int(claim_id.rsplit("-", 1)[1]))
    return {"results": [{"claim_id": claim_id, "started": started, "finished": datetime.now(timezone.utc)}]}

In [ ]:
claim_ids = [f"DEMO-{number}" for number in range(1, 7)]
timing_dispatcher = build_dispatcher(claim_ids, stand_in, timing_scores)

timings = []
for limit in [1, 3, 6]:
    result = timing_dispatcher.invoke({}, {"max_concurrency": limit})
    timings.append({"concurrency": limit, **result["scores"]})
pd.DataFrame(timings)

`seconds` is the interval from the first worker starting to the last finishing. `seconds_per_claim` is the mean worker duration, not elapsed time divided by the number of claims. These short sleeps demonstrate scheduling; their timings are not a performance benchmark.

## Full claim scores
The following functions preserve the lab's scoring rules. `score_table` takes reference answers explicitly rather than reading a global `labels` variable. Only the scorer receives those references. The claim worker receives a claim ID.

Decision accuracy checks the outcome alone. Accuracy also requires the payout within $1. Business mistake cost and model API cost are separate measures.

In [ ]:
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer: ClaimResult, expected: Answer) -> bool:
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer: ClaimResult, expected: Answer) -> float:
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0.0
    if answer["decision"] == "decline":
        return float(WRONGFUL_DECLINE)
    if answer["decision"] == "approve":
        return answer["payout"]
    return float(NEEDLESS_REFERRAL)


def score_table(results: list[ClaimResult], labels: dict[str, Answer]) -> pd.DataFrame:
    rows = []
    for answer in results:
        expected = labels[answer["claim_id"]]
        rows.append({
            "claim_id": answer["claim_id"],
            "handler": expected["decision"], "agent": answer["decision"],
            "handler_payout": expected["payout"], "agent_payout": answer["payout"],
            "right": is_right(answer, expected), "mistake_cost": mistake_cost(answer, expected),
            "llm_cost": answer["cost"], "started": answer["started"], "finished": answer["finished"],
            "seconds": (answer["finished"] - answer["started"]).total_seconds(),
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table: pd.DataFrame) -> RunScores:
    return {
        "decision_accuracy": float((table["agent"] == table["handler"]).mean()),
        "accuracy": float(table["right"].mean()),
        "cost_of_mistakes": float(table["mistake_cost"].sum()),
        "straight_through": float((table["agent"] != "refer").mean()),
        "llm_cost": float(table["llm_cost"].sum()),
        "seconds": (table["finished"].max() - table["started"].min()).total_seconds(),
        "slowest_claim": float(table["seconds"].max()),
        "seconds_per_claim": float(table["seconds"].mean()),
    }

In [ ]:
reference_answers: dict[str, Answer] = {
    "DEMO-1": {"decision": "approve", "payout": 100.0},
    "DEMO-2": {"decision": "approve", "payout": 200.0},
    "DEMO-3": {"decision": "approve", "payout": 300.0},
    "DEMO-4": {"decision": "decline", "payout": 0.0},
    "DEMO-5": {"decision": "decline", "payout": 0.0},
    "DEMO-6": {"decision": "refer", "payout": 0.0},
}
scripted_answers: dict[str, Answer] = {
    "DEMO-1": {"decision": "approve", "payout": 100.0},
    "DEMO-2": {"decision": "approve", "payout": 220.0},
    "DEMO-3": {"decision": "decline", "payout": 0.0},
    "DEMO-4": {"decision": "approve", "payout": 400.0},
    "DEMO-5": {"decision": "refer", "payout": 0.0},
    "DEMO-6": {"decision": "refer", "payout": 0.0},
}


def scripted_worker(claim_id: str) -> Dispatch:
    row = stand_in(claim_id)["results"][0]
    row.update(scripted_answers[claim_id])
    row.update(cost=0.01, rationale="Synthetic scoring fixture.")
    return {"results": [row]}


def quality_scores(results: list[ClaimResult]) -> RunScores:
    return run_scores(score_table(results, reference_answers))


scored_dispatcher = build_dispatcher(claim_ids, scripted_worker, quality_scores)
scored_result = scored_dispatcher.invoke({}, {"max_concurrency": 3})
pd.Series(scored_result["scores"])

The fixtures include a correct approval, an incorrect payout, a wrongful decline, a wrongful approval, an unnecessary referral and a correct referral. Expected decision accuracy is 3/6, full accuracy is 2/6, and mistake cost is $1,970: $20 + $1,500 + $400 + $50. The $0.06 API cost is synthetic; no money is spent.

## Offline checks
Check the graph with concurrency caps of 1, 3 and 6. The scorer must run once, after all six workers finish, and receive each claim exactly once. A simulated timeout must retry without duplicating results. A worker that exhausts its retries must fail the run without publishing a complete-run score.

In [ ]:
def check_dispatch(worker: Callable[[str], Dispatch], limit: int) -> Dispatch:
    scoring_calls: list[list[str]] = []

    def checked_scorer(results: list[ClaimResult]) -> RunScores:
        received = sorted(row["claim_id"] for row in results)
        assert received == claim_ids
        assert datetime.now(timezone.utc) >= max(row["finished"] for row in results)
        scoring_calls.append(received)
        return quality_scores(results)

    graph = build_dispatcher(claim_ids, worker, checked_scorer)
    result = graph.invoke({}, {"max_concurrency": limit})
    assert len(scoring_calls) == 1
    scores = result["scores"]
    assert isclose(scores["decision_accuracy"], 3 / 6)
    assert isclose(scores["accuracy"], 2 / 6)
    assert scores["cost_of_mistakes"] == 1970
    assert isclose(scores["straight_through"], 4 / 6)
    assert isclose(scores["llm_cost"], 0.06)
    return result


for limit in [1, 3, 6]:
    check_dispatch(scripted_worker, limit)

attempts: dict[str, int] = {}


def flaky_worker(claim_id: str) -> Dispatch:
    attempts[claim_id] = attempts.get(claim_id, 0) + 1
    if claim_id == "DEMO-6" and attempts[claim_id] == 1:
        raise TimeoutError("Simulated provider timeout")
    return scripted_worker(claim_id)


check_dispatch(flaky_worker, 3)
assert attempts["DEMO-6"] == 2
assert all(count == 1 for key, count in attempts.items() if key != "DEMO-6")

failed_attempts: list[str] = []
unexpected_scores: list[int] = []


def failing_worker(claim_id: str) -> Dispatch:
    if claim_id == "DEMO-6":
        failed_attempts.append(claim_id)
        raise TimeoutError("Persistent simulated timeout")
    return scripted_worker(claim_id)


def unexpected_scorer(results: list[ClaimResult]) -> RunScores:
    unexpected_scores.append(len(results))
    return quality_scores(results)


try:
    build_dispatcher(claim_ids, failing_worker, unexpected_scorer).invoke({}, {"max_concurrency": 3})
except TimeoutError:
    assert len(failed_attempts) == 3 and not unexpected_scores
else:
    raise AssertionError("An exhausted worker must fail the run")

base = datetime(2026, 1, 1, tzinfo=timezone.utc)
fixed_results: list[ClaimResult] = [
    {"claim_id": claim_id, **answer, "started": base, "finished": base + timedelta(seconds=index),
     "cost": 0.01, "rationale": "Fixed timing fixture."}
    for index, (claim_id, answer) in enumerate(scripted_answers.items(), 1)
]
fixed_scores = quality_scores(fixed_results)
assert fixed_scores["seconds"] == 6 and fixed_scores["slowest_claim"] == 6
assert fixed_scores["seconds_per_claim"] == 3.5
print("Offline checks passed: one complete score, correct metrics, successful retry and exhausted retry.")